# Task 2.2 Reproduction of Contribution

**Contribution Selected:** We are reproducing the central algorithmic contribution: the Deterministic Anti-Annealing Expectation Maximization (DAEM) method from Section 3.1. We aim to show its execution on the unbalanced toy dataset.

**Evaluation Metric:** We will evaluate the log-likelihood convergence and the parameter error directly (matching the cluster means).

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

def initialize_params(X, K):
    N, D = X.shape
    indices = np.random.choice(N, K, replace=False)
    means = X[indices]
    covs = [np.cov(X.T) for _ in range(K)]
    alphas = np.ones(K) / K
    return means, covs, alphas

This code block initializes the GMM parameters. We randomly pick 2 points as starting means, assign uniform mixing coefficients $\alpha_i = 1/K$, and use the aggregate dataset covariance for both components. 

*Corresponds to Section 3.1: 1. Initialize.*

In [2]:
def e_step(X, means, covs, alphas, beta):
    N, K = X.shape[0], len(means)
    r = np.zeros((N, K))
    
    for k in range(K):
        r[:, k] = alphas[k] * multivariate_normal.pdf(X, mean=means[k], cov=covs[k])
        
    r_beta = r ** beta
    r_beta_sum = r_beta.sum(axis=1)[:, np.newaxis] + 1e-10
    h_jt = r_beta / r_beta_sum
    return h_jt

This block computes the modified posterior probabilities $h_j(t)$ given the current $\beta$. The exponentiation by $\beta$ forces the assignment distribution to harden or soften based on temperature. 

*Corresponds to Section 3.1, Equation (2).*

In [3]:
def m_step(X, h_jt):
    N, D = X.shape
    K = h_jt.shape[1]
    Nk = h_jt.sum(axis=0)
    
    means_new = np.zeros((K, D))
    covs_new = []
    
    for k in range(K):
        means_new[k] = (1 / Nk[k]) * np.sum(h_jt[:, k][:, np.newaxis] * X, axis=0)
        
        diff = X - means_new[k]
        cov_k = (1 / Nk[k]) * np.dot((h_jt[:, k][:, np.newaxis] * diff).T, diff)
        cov_k += np.eye(D) * 1e-6
        covs_new.append(cov_k)
        
    alphas_new = Nk / N
    return means_new, covs_new, alphas_new

This code block formally updates the model parameters $\Theta$: means, covariances, and mixing coefficients $\alpha_k$ using the membership responsibilities computed in the E-step.

*Corresponds to Section 3.1: "M-step: estimate $\Theta^{(new)}$ using $h_j(t)$ values"*

In [4]:
def perturb_means(means):
    K, D = means.shape
    for k in range(K):
        noise = np.random.randn(D) * 0.1
        means[k] += noise
    return means

This function explicitly injects noise into the cluster means. Without this, merged clusters at low $\beta$ values would be structurally incapable of splitting symmetrically once $\beta$ exceeds 1.

*Corresponds to Section 3.1: "We must perturb the estimated parameters with a small noise term to enable the clusters to split..."*

In [5]:
def daem_fit(X, K, beta_schedule, iterations_per_beta=5):
    means, covs, alphas = initialize_params(X, K)
    for beta in beta_schedule:
        means = perturb_means(means)
        for _ in range(iterations_per_beta):
            h_jt = e_step(X, means, covs, alphas, beta)
            means, covs, alphas = m_step(X, h_jt)
    return means, covs, alphas

np.random.seed(42)
beta_schedule = [0.8, 1.0, 1.2, 1.5, 1.2, 1.0]
data_dir = 'data'
X = np.load(os.path.join(data_dir, 'X.npy'))
means_est, covs_est, alphas_est = daem_fit(X, 2, beta_schedule, iterations_per_beta=10)

print("Estimated Means:\n", means_est)
print("Estimated Alphas:\n", alphas_est)

Estimated Means:
 [[-0.62373317  2.56144926]
 [ 0.15826621 -0.17507395]]
Estimated Alphas:
 [0.0829375  0.91706247]
